# 02 · 실험 템플릿

새 아이디어를 만들 때 **이 노트북을 복사해서** 쓴다.

## 반드시 기억할 세 가지

1. **폴드2024 와 폴드2023 의 하이퍼파라미터 순위는 Spearman −0.806 으로 역전된다.**
   27개 조합 중 두 폴드 동시 개선은 **0개**였다. 그래서 `채택후보` 판정이 나와도
   LB 개선이 보장되지 않는다 — 어디까지나 후보다.
2. **CV 이득을 LB 기대치로 옮기지 마라.** 실측 전이율은 v8 0.21배, v9 0.53배,
   HPO 0.06배였다.
3. **블렌드 가중치는 CV 로 고르지 마라.** 같은 모델에서 CV +14.2 인 설정이
   LB 에서 −11.66 이었다 (부호가 반대).

In [ ]:
# ── 부트스트랩 · 이 셀을 가장 먼저 실행 ──────────────────────────
# 코랩은 노트북마다 런타임(VM)이 다르다. 01 에서 받아둔 것은 여기 없다.
# 이 셀 하나가 레포 클론 → 의존성 → 데이터 확보까지 전부 처리한다.
#   데이터는 로컬 → Drive → 데이터 서버 순으로 찾고, 서버에서 받은 건 Drive 에 백업한다.
RUNNER_NAME = "본인이름"        # ← 여기만 바꾼다

import os, sys, subprocess
if not os.path.exists('/content/lga-repo/src/config.py'):
    subprocess.run(['git','clone','-q',
                    'https://github.com/hyunku9566/lga_data.git','/content/lga-repo'])
else:
    subprocess.run(['git','-C','/content/lga-repo','pull','-q'])
subprocess.run(['pip','install','-q','-r','/content/lga-repo/requirements-colab.txt'])

for m in [k for k in list(sys.modules)
          if k.startswith('src') or k in ('config','lib_lga','experiment','download','bootstrap')]:
    del sys.modules[m]
for _p in ('/content/lga-repo/src', '/content/lga-repo'):
    if _p not in sys.path: sys.path.insert(0, _p)

# 서버에서 받아야 할 수도 있으니 비밀번호를 미리 받아둔다 (Drive 에 있으면 안 쓴다)
if not os.environ.get('LGA_PASSWORD'):
    from getpass import getpass
    os.environ['LGA_PASSWORD'] = getpass('team 비밀번호 (Drive 에 캐시 있으면 엔터): ').strip()

from bootstrap import setup
C = setup(runner=RUNNER_NAME)
import lib_lga as L, experiment as E


## 기준선

모든 아이디어는 **하나의 고정 기준선**과 비교된다 (v7 120피처 · XGB d6/mcw1500/n600 · 시드5).
`ledger/baseline.json` 에 캐시돼 있어 매번 다시 돌지 않는다.

In [ ]:
base = E.get_baseline()
print(f"기준선  2024 {base['m24']:.1f} ±{base['m24_sd']:.1f}   2023 {base['m23']:.1f} ±{base['m23_sd']:.1f}")
print(f"시드 {base['seeds']} 기준 노이즈 막대(2se, 실험 시드3): "
      f"±{2*E._se(base['m24_sd'],3):.1f} / ±{2*E._se(base['m23_sd'],3):.1f}")

## 예시 A — 하이퍼파라미터

`grid` 에 넣은 조합이 모두 평가되고, **조합마다 원장에 즉시 기록**된다.
런타임이 끊겨도 다시 실행하면 이미 끝난 조합은 건너뛴다.

In [ ]:
RUNNER_NAME = '본인이름'      # 원장 파일이 팀원별로 갈린다

df = E.run_experiment(
    name='예시_depth',
    kind='hparam',
    grid={'max_depth': [6, 8]},
    seeds=2,
    runner=RUNNER_NAME,
    notes='템플릿 예시',
)
df[['params_json','m24','m23','delta24','delta23','verdict']]

## 예시 B — 새 피처

`feature_fn(RAW, X98, base) -> DataFrame` 를 주면 반환한 열이 기존 120피처에 붙는다.
행 수와 순서는 `RAW` 와 같아야 한다.

아래는 **투수 커리어 대비 당해 성적 편차**를 넣어보는 예다.
(참고: 비슷한 계열인 G2 '커리어 기준 축소' 는 튜닝 전 +2.6 이었으나
튜닝 후에는 +1.9 로 줄었다 — 모델이 좋아지면 피처 이득이 줄어드는 경향이 있다.)

In [ ]:
import numpy as np, pandas as pd

def my_feature(RAW, X98, b):
    # 앵커(시즌 첫 행)를 빼서 '당해 누적' 을 복원하고 커리어와 비교한다
    nc, rc = 'asof_pitcher_n', 'asof_pitcher_success_rate'
    t = RAW[['pitcher_id','season',nc,rc]].copy()
    t['succ'] = t[nc] * t[rc].fillna(0)
    S = t.loc[t.groupby(['pitcher_id','season'])[nc].idxmin()].set_index(['pitcher_id','season'])[[nc,'succ']]
    a = RAW[['pitcher_id','season']].join(S, on=['pitcher_id','season'])
    an, asu = a[nc].fillna(0).values, a['succ'].fillna(0).values
    dn = np.maximum(RAW[nc].values - an, 0)
    ds = np.maximum(np.nan_to_num(RAW[nc].values * RAW[rc].values) - asu, 0)
    car = np.where(an > 0, asu / np.maximum(an, 1), np.nan)
    ssn = np.where(dn > 0, ds / np.maximum(dn, 1), np.nan)
    return pd.DataFrame({'my_ssn_minus_car': (ssn - car).astype('float32'),
                         'my_trust': (dn / (dn + 200.)).astype('float32')}, index=RAW.index)

df = E.run_experiment(
    name='예시_커리어대비편차',
    kind='feature',
    feature_fn=my_feature,
    seeds=2,
    runner=RUNNER_NAME,
    notes='템플릿 예시',
)

## 결과 확인

원장은 `ledger/ledger_<이름>.csv` 에 쌓인다. 집계는 `04_summary.ipynb` 에서 한다.

In [ ]:
led = E.read_all_ledgers()
print(f'전체 {len(led)}건')
if len(led):
    display(led.tail(10)[['runner','name','params_json','delta24','delta23','verdict']])